In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install gekko
!pip install xlsxwriter
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from gekko import GEKKO
from datetime import datetime
from scipy.sparse import coo_matrix, eye, hstack
from sklearn.metrics import mean_squared_error
mse=mean_squared_error
class Natural_gas_flow:

    def __init__(self, data, disp=False, weymouth_type='MPCC',
                 num_approximations=20,initial=None):
        """
        In the initialization,  the xlsx file that contains the network data is
        loaded, the information is extracted from this same file, the incidence
        matrix is built, which describes the topology of the system and finally the
        upper and lower limits are established in each element of the system.
        """
        self.data = data
        self.disp = disp
        self.num_approximations = num_approximations
        self.weymouth_type = weymouth_type
        # The information is extracted from the xlsx file
        self.node_info = pd.read_excel(self.data, sheet_name='node.info')
        self.node_info['node_id'] = self.node_info['node_id'].astype('int')
        self.node_info['type'] = self.node_info['type'].astype('int')

        self.node_dem = pd.read_excel(self.data, sheet_name='node.dem')
        self.node_dem['Total'] = self.node_dem.sum(axis=1)
        self.node_user = self.node_dem[self.node_dem['Total'] != 0]
        self.loads = self.node_dem['Total'].values
        self.node_demcost = pd.read_excel(self.data, sheet_name='node.demcost')

        self.well = pd.read_excel(self.data, sheet_name='well')
        self.well['node'] = self.well['node'].astype('int')

        self.node_deminitial = pd.read_excel(self.data, sheet_name='node.deminitial')

        self.pipe = pd.read_excel(self.data, sheet_name='pipe')
        self.pipe['fnode'] = self.pipe['fnode'].astype('int')
        self.pipe['tnode'] = self.pipe['tnode'].astype('int')

        self.comp = pd.read_excel(self.data, sheet_name='comp')
        self.comp['fnode'] = self.comp['fnode'].astype('int')
        self.comp['tnode'] = self.comp['tnode'].astype('int')
        self.comp['Type'] = self.comp['Type'].astype('int')

        self.sto = pd.read_excel(self.data, sheet_name='sto')
        self.sto['node'] = self.sto['node'].astype('int')
        try:
            self.coordinates = pd.read_excel(self.data, sheet_name='coordinates')
        except:
            self.coordinates = False
        self.max_ratio = self.comp['ratio']
        self.N = len(self.node_info)
        self.W = len(self.well)
        self.P = len(self.pipe)
        self.C = len(self.comp)
        self.UNS = len(self.node_demcost) * len(self.node_demcost.T)
        self.S = len(self.sto)
        self.Kij = self.pipe['Kij'].values
        N = len(self.node_info)
        # Incidence Matrix is created
        # Wells ok
        df_wells = pd.read_excel(self.data, sheet_name='well')
        W = len(self.well)
        wells = coo_matrix(
            (np.ones(W, ), (self.well['node'] - 1, np.arange(W))), shape=(N, W))
        # Pipes ok
        df_pipes = pd.read_excel(self.data, sheet_name='pipe')

        P = len(self.pipe)
        data = np.concatenate((-1.0 * np.ones(P), np.ones(P)))
        row = pd.concat((self.pipe['fnode'] - 1, self.pipe['tnode'] - 1))
        col = np.concatenate(2 * [np.arange(P)])

        pipes = hstack(
            2 * [coo_matrix((data, (row, col)), shape=(N, P))]).toarray()
        # print('Pipes:', pipes.shape)
        # Compressors ok
        C = len(self.comp)
        data = np.concatenate((-1.0 * np.ones(C), np.ones(C)))
        row = pd.concat((self.comp['fnode'] - 1, self.comp['tnode'] - 1))
        col = np.concatenate(2 * [np.arange(C)])
        comps = coo_matrix((data, (row, col)), shape=(N, C))
        # Users ok
        users = hstack(len(self.node_demcost.T) * [eye(N)])
        # Storage
        self.sto = pd.read_excel(self.data, sheet_name='sto')
        S = len(self.sto)
        sto = coo_matrix(
            (np.ones(S, ), (self.sto['node'] - 1, np.arange(S))), shape=(N, S))
        sto = hstack([sto, -1.0 * sto])

        self.Minc = (hstack((wells, pipes, comps, users, sto))).toarray()
        # The limits of the elementes of the network are stablished
        fp = self.pipe['FG_O'] * (self.pipe['FG_O'] > 0)
        fn = self.pipe['FG_O'] * (self.pipe['FG_O'] < 0)
        f_ext = np.concatenate((fp, fn))

        # print(np.array(self.node_deminitial).flatten(order='F').shape)
        self.lb = np.concatenate((self.well['Imin'].values,
                                  [0] * len(self.pipe),
                                  self.pipe['Fg_min'].values,
                                  [0] * len(self.comp),
                                  [0] * len(self.node_dem) *
                                  (len(self.node_dem.T) - 1),
                                  [0] * len(self.sto) * 2,
                                  self.node_info['Pmin'].values))
        if np.any(initial==None):
            self.initial = np.concatenate((self.well['I'].values,
                                       f_ext,
                                       self.comp['fgc'].values,
                                       np.array(self.node_deminitial).flatten(order='F'),
                                       [0] * len(self.sto.values),
                                       [0] * len(self.sto.values),
                                       self.node_info['p'].values))
        else:
            try:
                if initial.shape[0]==self.lb.shape[0]:
                    self.initial=initial
                else:
                    self.initial = np.concatenate((self.well['I'].values,
                                       f_ext,
                                       self.comp['fgc'].values,
                                       np.array(self.node_deminitial).flatten(order='F'),
                                       [0] * len(self.sto.values),
                                       [0] * len(self.sto.values),
                                       self.node_info['p'].values))
            except:
                self.initial = np.concatenate((self.well['I'].values,
                                       f_ext,
                                       self.comp['fgc'].values,
                                       np.array(self.node_deminitial).flatten(order='F'),
                                       [0] * len(self.sto.values),
                                       [0] * len(self.sto.values),
                                       self.node_info['p'].values))



        self.ub = np.concatenate((self.well['Imax'].values,
                                  self.pipe['Fg_max'].values,
                                  [0] * len(self.pipe),
                                  self.comp['fmaxc'],
                                  self.node_dem['Res'],
                                  self.node_dem['Ind'],
                                  self.node_dem['Com'],
                                  self.node_dem['NGV'],
                                  self.node_dem['Ref'],
                                  self.node_dem['Pet'],
                                  self.sto['V0'] - self.sto['Vmax'].values,
                                  self.sto['Vmax'] - self.sto['V0'].values,
                                  self.node_info['Pmax'].values))

        self.objective_function()

    def objective_function(self):
        # Initialize Model
        self.m = GEKKO(remote=False)
        self.m.options.SOLVER =3 # IPOPT Solver

        cost = np.concatenate((self.well['Cg'].values,
                               self.pipe['C_O'].values,
                               -1 * self.pipe['C_O'].values,
                                self.comp['costc'].values,
                               self.node_demcost['al_Res'].values,
                               self.node_demcost['al_Ind'].values,
                               self.node_demcost['al_Com'].values,
                               self.node_demcost['al_NGV'].values,
                               self.node_demcost['al_Ref'].values,
                               self.node_demcost['al_Pet'].values,
                               (self.sto['C_S+'] - self.sto['C_V']).values,
                               -1 * (self.sto['C_S-'] -
                                     self.sto['C_V']).values,
                               self.sto['C_V'].values))

        NV = (len(self.well) + 2 * len(self.pipe) + len(self.comp) +
              (len(self.node_demcost) * len(self.node_demcost.T)) +
              2 * len(self.sto) + len(self.node_info))

        self.NV_obj = NV - len(self.node_info)

        limits = (self.initial, self.lb, self.ub)

        x = [self.m.Var(value=value,lb=lb,ub=ub) for value, lb, ub in zip(self.initial, self.lb, self.ub)]
        # x = [self.m.Var(lb=lb, ub=ub) for lb, ub in zip(self.lb, self.ub)]
        self.X = np.array(x)


        self.f_plus = [self.m.Var(lb=lb, ub=ub) for lb, ub in zip([0]*len(self.pipe), self.pipe['Fg_max'].values)]
        self.f_minus = [self.m.Var(lb=lb, ub=ub) for lb, ub in zip([0]*len(self.pipe), self.pipe['Fg_max'].values)]
        self.y = [self.m.Var() for i in self.pipe['Fg_max']]
        # for i, value in enumerate(np.array(self.initial)):
        #     self.m.fix_initial(x[i], value)

        # self.X = np.array(x)

        # B_lb = np.array([0] * 2 * len(self.pipe))
        # B_ub = np.array([1] * 2 * len(self.pipe))

        # B = [self.m.Var(lb=lb, ub=ub, integer=True)
        #      for lb, ub in zip(B_lb, B_ub)]
        # self.B = np.array(B)

        # for x, x0 in zip(self.X, self.initial):
        #   self.m.fix_initial(x, x0)

        self.J = cost.T @ np.concatenate((self.X[:self.NV_obj],
                                          self.sto['V0'].values))

        self.constraints()
        self.solve_network()

    def constraints(self, ):
        self.gas_balance()
        self.comp_ratio()

        if self.weymouth_type == 'MPCC':
            self.weymouth_MPCC()
        elif self.weymouth_type == 'SOC':
            self.weymouth_SOC()
        elif self.weymouth_type == 'Taylor':
            self.weymouth_Taylor()
        elif self.weymouth_type == 'tanh':
            self.weymouth_tanh()
    # (2.24)
    def gas_balance(self):
        ftrans_max = self.pipe['Fg_max'].values
        pipe_indx1 = len(self.well)
        pipe_indx2 = len(self.well) + len(self.pipe)

        # self.m.Equations([self.X[pipe_indx1+i] + self.X[pipe_indx2+i] <= j for i, j in enumerate(ftrans_max)])
        #self.m.Equations([self.net_flow(self.X)[i] <=
        #                 j for i, j in enumerate(ftrans_max)])

        self.A = self.X[:self.NV_obj] @ self.Minc.T
        node=int(len(self.node_dem))
        for i in range(node):
            load = self.m.Param(value=self.loads[i])
            self.m.Equation(self.A[i] == load)

    def gas_balance_eval(self):
        sol = self.X[:self.NV_obj]
        self.x_sol = np.array([val.value[0] for val in sol])
        self.eval = self.x_sol @ self.Minc.T - self.loads
        return self.eval

    def comp_ratio(self):
        press = self.X[-self.N:]
        subA = self.Minc[:, self.W + 2 * self.P:self.W + 2 * self.P + self.C]
        g = np.prod(press.reshape(-1, 1) ** subA, 0)
        g2 = self.m.Param(value=1)

        for i, constrain in enumerate(g):
            g1 = self.m.Param(value=self.max_ratio[i])
            self.m.Equation(constrain <= g1)
            self.m.Equation(constrain >= g2)

    def net_flow(self, x):
        flow = x[self.W:self.W + self.P] + \
            x[self.W + self.P:self.W + 2 * self.P]
        return flow

    def weymouth_MPCC(self):
        f = (self.net_flow(self.X).reshape(-1,))
        press = np.array(self.X[-self.N:])
        press2 = np.array(self.X[-self.N:]) ** 2

        for i in range(self.P):

            i_index = (self.pipe['fnode'] - 1)[i]
            j_index = (self.pipe['tnode'] - 1)[i]
            p = press2[i_index] - press2[j_index]
            K = self.Kij[i]#f[i]== self.m.sign2(p) * self.m.sqrt( K* self.m.abs2(p))
            self.m.Equation(f[i]== self.m.sign2(p) * self.m.sqrt( K* self.m.abs2(p)))#self.m.sign2(f[i]) * f[i]**2 == p*K)

    def weymouth_tanh(self):
        f = (self.net_flow(self.X).reshape(-1,))
        press = np.array(self.X[-self.N:])
        press2 = np.array(self.X[-self.N:]) ** 2

        for i in range(self.P):

            i_index = (self.pipe['fnode'] - 1)[i]
            j_index = (self.pipe['tnode'] - 1)[i]
            p = press2[i_index] - press2[j_index]
            K = self.Kij[i]
            self.m.Equation(self.m.tanh(f[i]) * f[i]**2 == p*K)

    def weymouth_SOC(self):
        print('SOC')
        f = self.net_flow(self.X).reshape(-1,)
        press = self.X[-self.N:]
        Minc_P = self.Minc[:, self.W:self.W+self.P]
        Minc_P_i = Minc_P.copy()
        Minc_P_j = Minc_P.copy()
        Minc_P_i[Minc_P_i > 0] = 0
        Minc_P_i = -1*Minc_P_i
        Minc_P_j[Minc_P_j < 0] = 0
        p_min = self.node_info['Pmin']
        p_max = self.node_info['Pmax']
        p_min_i = p_min @ (Minc_P_i)
        p_max_j = p_max @ (Minc_P_j)
        p_max_i = p_max @ (Minc_P_i)
        p_min_j = p_min @ (Minc_P_j)
        phi_plus_lowlimit = p_min @ np.abs(Minc_P)
        phi_plus_uplimit = p_max @ np.abs(Minc_P)
        phi_minus_lowlimit = p_min_i - p_max_j
        phi_minus_uplimit = p_max_i - p_min_j

        phi_plus = (press @ np.abs(Minc_P))
        phi_minus = (press @ (-1*Minc_P))


        y = [self.m.Var(integer=True) for i in range(self.P)]
        Phi = [self.m.Var() for i in range(self.P)]
        M = [self.m.Var(lb=1) for i in range(self.P)]


        Phi_lowlimit1 = (phi_plus_lowlimit * phi_minus) + \
                    (phi_plus * phi_minus_lowlimit) - \
                    (phi_plus_lowlimit * phi_minus_lowlimit)

        Phi_lowlimit2 = (phi_plus_uplimit * phi_minus) + \
                    (phi_minus * phi_minus_uplimit) - \
                    (phi_plus_uplimit * phi_minus_uplimit)


        Phi_uplimit1 = (phi_plus_uplimit * phi_minus) + \
                    (phi_plus * phi_minus_lowlimit) - \
                    (phi_plus_uplimit * phi_minus_lowlimit)

        Phi_uplimit2 = (phi_plus_lowlimit * phi_minus) + \
                    (phi_plus * phi_minus_uplimit) - \
                    (phi_plus_lowlimit * phi_minus_uplimit)

        for i in range(self.P):
            self.m.Equation(phi_plus[i] <= self.m.Param(value = phi_plus_uplimit[i]))
            self.m.Equation(phi_plus[i] >= self.m.Param(value = phi_plus_lowlimit[i]))
            self.m.Equation(phi_minus[i] <= self.m.Param(value = phi_minus_uplimit[i]))
            self.m.Equation(phi_minus[i] >= self.m.Param(value = phi_minus_lowlimit[i]))

            self.m.Equation(Phi[i] <= self.m.Param(value = Phi_uplimit1[i]))
            self.m.Equation(Phi[i] <= self.m.Param(value = Phi_uplimit2[i]))
            self.m.Equation(Phi[i] >= self.m.Param(value = Phi_lowlimit1[i]))
            self.m.Equation(Phi[i] >= self.m.Param(value = Phi_lowlimit2[i]))

            self.m.Equation(self.m.Param(value = f[i]) <= self.Kij[i] * Phi[i] + \
                            M[i]**2 * (1 - y[i]))

            self.m.Equation(self.m.Param(value = f[i]) <= -1*self.Kij[i] * Phi[i] + \
                            M[i]**2 * y[i])

    def weymouth_Taylor(self):
        print('Taylor')
        num_pipes = len(self.pipe)
        num_approximations = self.num_approximations
        press = self.X[-self.N:]
        f = self.net_flow(self.X).reshape(-1,)

        for i in range(num_pipes):
            low_limit = self.node_info['Pmin'].values[i]
            upp_limit = self.node_info['Pmax'].values[i]
            K = self.Kij[i]
            pi = press[self.pipe['fnode'][i] - 1]
            pj = press[self.pipe['tnode'][i] - 1]
            g_value = self.m.Param(value = f[i])

        for j in range(num_approximations):
            PO, PI = np.sort(np.random.uniform(low=low_limit, high=upp_limit, size=(2,)))
            g = K*( (PI/np.sqrt(PI**2-PO**2))*pi + (PO/np.sqrt(PI**2-PO**2))*pj)
            self.m.Equation(self.m.abs(g_value) <= g)



    def weymouth_eval(self):
        sol = self.X[:self.NV_obj]
        self.x_sol = np.array([val.value[0] for val in sol])
        f_plus = self.x_sol[self.W: self.W+self.P]
        f_minus = self.x_sol[self.W+self.P : self.W+self.P+self.P]

        f_net = f_plus + f_minus
        p2 = np.array([ (p[0])**2 for p in self.X[-self.N:]])
        self.pipe_f_node = self.pipe['fnode'] - 1
        self.pipe_t_node = self.pipe['tnode'] - 1
        K = self.pipe['Kij'].values
        p2_sub = p2[self.pipe_f_node] - p2[self.pipe_t_node]
        self.wey_eval = (K * p2_sub) - np.sign(f_net) * f_net**2

        return self.wey_eval

    def solve_network(self):
        self.m.options.OTOL = 1e-7
        self.m.options.MAX_ITER = 1e9
        self.m.solver_options = ['mu_strategy adaptive',
                                 'constr_viol_tol 1e-7',
                                 'acceptable_tol 1e-7', ]
        #  'bound_push 1e-10',\
        #  'bound_frac 1e-10']
        self.m.Minimize(self.J)
        #self.m.open_folder()
        a=self.m.solve(disp=self.disp)
    def get_values(self):
        N = self.W + self.P + self.C + self.UNS + self.N
        names = [0] * N
        values = [0] * N

        for i in range(self.W):
            inj = self.well['node'].values[i]
            names[i] = 'Well ' + str(inj)
            values[i] = self.X[i].value[0]

        f_plus = self.X[self.W: self.W + self.P]
        f_minus = self.X[self.W + self.P: self.W + self.P + self.P]
        for i in range(self.P):
            ind = self.W + i
            From = self.pipe['fnode'][i]
            To = self.pipe['tnode'][i]
            names[ind] = 'Net flow pipe ' + str(From) + '-' + str(To)
            values[ind] = f_plus[i][0] + f_minus[i][0]

        for i in range(self.C):
            ind = self.W + self.P + i
            From = self.comp['fnode'][i]
            To = self.comp['tnode'][i]
            names[ind] = 'Flow compressor ' + str(From) + '-' + str(To)
            values[ind] = self.X[self.W + 2 * self.P + i].value[0]

        k = 0
        for i in range( len(self.node_demcost.T)):
            for j in range(len(self.node_demcost)):
                ind = self.W + self.P + self.C + k
                names[ind] = 'Shortage in node ' + str(j+1) + '-' + 'Load ' + str(i+1)
                values[ind] = self.X[self.W + 2 * self.P + self.C+ k].value[0]
                k += 1

        for i in range(self.N):
            ind = self.W + self.P + self.C + self.UNS + i
            names[ind] = 'Press in node  ' + str(i+1)
            values[ind] = self.X[-self.N+i].value[0]
        names.insert(0, 'Objective')
        values.insert(0, self.m.options.objfcnval)
        self.values=values
        return [self.m.options.SOLVETIME,self.values]
    def Evaluate(self):
        y_pred=self.weymouth_eval()
        W=mse(np.zeros(y_pred.shape),y_pred)
        y_pred=self.gas_balance_eval()
        B=mse(np.zeros(y_pred.shape),y_pred)
        S=self.X[-8:]
        #return  self.max_ratio
        ratio=np.zeros((2))
        ratio[0]=np.maximum(0,S[2][0]-self.max_ratio[0]*S[1][0])
        ratio[0]=np.maximum(0,S[3][0]-self.max_ratio[0]*S[2][0])
        ratio=mse(np.zeros(ratio.shape),ratio)
        return [B,W,ratio,self.get_values()]
def GenW(Data_inf,Data_W,Data_T,Data_C,Data_Dc,Data_sto):
    N = len(Data_inf)
    W = len(Data_W)
    wells = coo_matrix((np.ones(W, ), (Data_W['node'] - 1, np.arange(W))), shape=(N, W))
    P = len(Data_T)
    data = np.concatenate((-1.0 * np.ones(P), np.ones(P)))
    row = pd.concat((Data_T['fnode'] - 1, Data_T['tnode'] - 1))
    col = np.concatenate(2 * [np.arange(P)])

    pipes = coo_matrix((data, (row, col)), shape=(N, P))

    C = len(Data_C)
    data = np.concatenate((-1.0 * np.ones(C), np.ones(C)))
    row = pd.concat((Data_C['fnode'] - 1, Data_C['tnode'] - 1))
    col = np.concatenate(2 * [np.arange(C)])
    comps = coo_matrix((data, (row, col)), shape=(N, C))
    users=hstack(len(Data_Dc.T) * [eye(N)])
    S = len(Data_sto)
    sto = coo_matrix((np.ones(S, ), (Data_sto['node'] - 1, np.arange(S))), shape=(N, S))
    sto = hstack([sto, -1.0 * sto])
    w=(hstack((wells, pipes, comps, users, sto))).toarray()
    return w

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 3.2 MB/s eta 0:00:00


In [3]:
from gekko import GEKKO
m = GEKKO(remote=True)            # create GEKKO model
x = m.Var()            # define new variable, default=0
y = m.Var()            # define new variable, default=0
m.Equations([3*x+2*y==1, x+2*y==0])
m.options.SOLVER=3 # equations
m.solve(disp=False,)    # solve
print(x.value,y.value)

[0.5] [-0.25]


In [ ]:
U = np.load('/content/drive/Shareddrives/red_gas_col/Prueba/Data/Sintéticos/' + 'U.npy')

In [ ]:
U.shape

(100, 500, 5)

In [ ]:
def GenW(Data_inf,Data_W,Data_T,Data_C,Data_Dc,Data_sto):
    N = len(Data_inf)
    W = len(Data_W)
    wells = coo_matrix((np.ones(W, ), (Data_W['node'] - 1, np.arange(W))), shape=(N, W))
    P = len(Data_T)
    data = np.concatenate((-1.0 * np.ones(P), np.ones(P)))
    row = pd.concat((Data_T['fnode'] - 1, Data_T['tnode'] - 1))
    col = np.concatenate(2 * [np.arange(P)])

    pipes = coo_matrix((data, (row, col)), shape=(N, P))

    C = len(Data_C)
    data = np.concatenate((-1.0 * np.ones(C), np.ones(C)))
    row = pd.concat((Data_C['fnode'] - 1, Data_C['tnode'] - 1))
    col = np.concatenate(2 * [np.arange(C)])
    comps = coo_matrix((data, (row, col)), shape=(N, C))
    users=hstack(len(Data_Dc.T) * [eye(N)])
    S = len(Data_sto)
    sto = coo_matrix((np.ones(S, ), (Data_sto['node'] - 1, np.arange(S))), shape=(N, S))
    sto = hstack([sto, -1.0 * sto])
    w=(hstack((wells, pipes, comps, users, sto))).toarray()
    return w


In [ ]:
import time
from scipy.io import savemat, loadmat
from tqdm import tqdm
from io import BytesIO
from IPython import display
import scipy.io as sio
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from tqdm import tqdm
from datetime import datetime
non=0
patht='/kaggle/input/datatest/'
i=6
nodes=[8,9,10,11,12,13,14,15,18]
node=nodes[i]
si=np.load(patht+'Red'+str(i)+'.npy')
pnodes=['ng_case8.xlsx','ng_case9.xlsx','ng_case10.xlsx','ng_case11.xlsx','ng_case12.xlsx','ng_case13.xlsx','ng_case14.xlsx','ng_case15.xlsx','ng_caseCol_18.xlsx']
path = patht+pnodes[i]
#Inp=loadmat('/content/drive/Shareddrives/Maestría/Optimización/Data/SNR_None/Inputs/inputs_1.mat')['inputs']
data = pd.ExcelFile(path)
Data_D=pd.read_excel(path,sheet_name='node.dem')
inputs=[]
outputs=[]
index=[]
tim=[]
Data_inf=pd.read_excel(path,sheet_name='node.info')
Data_D=pd.read_excel(path,sheet_name='node.dem')
Data_Dc=pd.read_excel(path,sheet_name='node.demcost')
Data_W=pd.read_excel(path,sheet_name='well')
Data_T=pd.read_excel(path,sheet_name='pipe')
Data_C=pd.read_excel(path,sheet_name='comp')
Data_sto=pd.read_excel(path,sheet_name='sto')
Cost=np.concatenate((Data_W['Cg'].values,Data_T['C_O'].values,Data_C['costc'],Data_Dc['al_Res'].values,
                Data_Dc['al_Ind'].values,Data_Dc['al_Com'].values,Data_Dc['al_NGV'].values,Data_Dc['al_Ref'].values,Data_Dc['al_Pet'].values,
                    Data_sto['C_S+'].values-Data_sto['C_V'].values,-1*(Data_sto['C_S-']-Data_sto['C_V']).values,Data_sto['C_V'].values)).reshape(-1,1)
cost=Cost
w=GenW(Data_inf,Data_W,Data_T,Data_C,Data_Dc,Data_sto)
pbar = tqdm(total=100)
Wey=[]
Bal=[]
Com=[]
i=Data_T['fnode'].values.astype(int)-1
j=Data_T['tnode'].values.astype(int)-1
CK=Data_T['Kij'].values.reshape(-1,)
nod=len(Data_inf)
ic = Data_C['fnode'].values.astype(int) - 1
jc = Data_C['tnode'].values.astype(int) - 1
Bc = Data_C['ratio'].values
from sklearn.metrics import mean_squared_error,mean_absolute_error,mean_absolute_percentage_error
mape=mean_absolute_percentage_error
ban=0
for jd in range(5,6):
    outputs=[]
    Wey.append([])
    Bal.append([])
    Com.append([])
    for id1 in range(100):#for j in range(muestras):
        #np.random.seed(3)
        load_data= si[jd][id1]
        #data = pd.ExcelFile(path)
        zero_load = np.zeros(load_data.shape)
        node_dem = pd.DataFrame({'Res': load_data, #+ normal_loads,
                                'Ind': zero_load,
                                'NGV': zero_load,
                                'Com': zero_load,
                                'Pet': zero_load,
                                'Ref': zero_load})
        sheet_names = data.sheet_names
        sheets = dict()
        for ik in sheet_names:
            sheets[ik] = pd.read_excel(data, sheet_name=ik)
        sheets['node.dem'] = node_dem
        import io
        output = io.BytesIO()

        with pd.ExcelWriter(output, engine='xlsxwriter') as writer:
            for ik in sheets:
                sheets[ik].to_excel(writer, sheet_name=ik, index=False)

        # Retrieve the bytes from the BytesIO buffer
        excel_bytes = output.getvalue()
        data = pd.ExcelFile(BytesIO(excel_bytes))
        disp =False
        try:
            p= Natural_gas_flow(data, disp=disp, weymouth_type='MPCC',initial=None)
            Out=p.get_values()

            F=np.array(Out[1])[1:-node]
            F =np.concatenate((F, np.zeros((2,))), axis=0)
            P=np.array(Out[1])[-node:]
            Ft=F[1:1+len(Data_T)]
            Pi=P[i]
            Pj=P[j]
            Pt=np.sign(Pi**2-Pj**2)*np.sqrt(CK*np.abs(Pi**2-Pj**2))
            are=are = P[jc] / P[ic]
            ind1 = np.unique(np.concatenate((np.where(~(are[0] <= Bc[0]) | ~(are[0] >= 1.0))[
                                # .shape[0]/are.shape[0]*100
                                0], np.where(~(are[1] <= Bc[1]) | ~(are[1] >= 1.0))[0])))
            re2 = np.zeros(1,)
            if len(ind1) != 0:
                re2[ind1] = 100
            Wey[ban].append(mape(Ft,Pt))
            Bal[ban].append(mape(load_data,F@w.T))
            Com[ban].append(mape(np.zeros(1),re2))

        except:
            1+2
            non=non+1
        pbar.update(1)
    ban=ban+1
pbar.close()
savepath='/kaggle/working/'
savemat(savepath+'Wey'+str(node)+'6'+'.mat', mdict={'inputs': Wey})
savemat(savepath+'Bal'+str(node)+'6'+'.mat', mdict={'inputs': Bal})
savemat(savepath+'Com'+str(node)+'6'+'.mat', mdict={'inputs': Com})


 64%|██████▍   | 64/100 [32:13<20:16, 33.79s/it]